# Carta Social: cleaning and geocoding

This workflow standardizes service records, prepares one address per equipment,
and associates coordinates from a matching cache or an explicitly selected
geocoding provider.

Geocoding converts an address into an estimated location. A returned coordinate
may refer to a building, street, or broader area; availability alone does not
establish positional accuracy.

Transformation logic is implemented in `scripts/cleaning.py` and
`scripts/geocoder.py`. The default configuration makes no external requests.


In [ ]:
from pathlib import Path
import sys

# Support execution from the repository root or the notebooks directory.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "scripts" / "utils.py").exists():
    raise FileNotFoundError("Start Jupyter from the ScraperCartaSocial repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
import os
import pandas as pd

from scripts.utils import RAW, INTERIM, INPUTS, ensure_data_directories, load_environment
from scripts.cleaning import clean_services, prepare_addresses, select_services, standardize_columns
from scripts.geocoder import load_coordinate_cache, geocode_addresses

load_environment()
ensure_data_directories()


## Service-table preparation

Source labels are mapped to English field names, surrounding whitespace is
removed, and empty strings become missing values. Capacity and user counts
must be non-negative integers; dates follow `YYYY-MM-DD`.

Invalid non-missing values receive explicit flags. Missing counts are not
replaced with zero, because an unknown count differs from an observed zero.
All source rows are retained.


In [ ]:
source_file = INPUTS["services"]
services_raw = pd.read_csv(source_file, dtype={"Equipment ID": "string"})
services_clean = clean_services(services_raw)
services_clean.head()


In [ ]:
print("Source rows:", len(services_raw))
print("Rows after cleaning:", len(services_clean))
print("Invalid capacity values:", services_clean["capacity_invalid"].sum())
print("Invalid user counts:", services_clean["users_invalid"].sum())

services_clean.to_csv(INTERIM / "services_clean.csv", index=False)


## Service selection and address universe

The default processes all service types in the configured CSV. Set service_types
to a list of exact source labels to select any categories. Available labels
are printed below; processing modules do not hardcode a category list.

Optional workbook mode uses inputs.selection_workbook, inputs.service_sheet,
and inputs.address_sheet from config.toml. Set use_workbook_selection to True
only when those inputs are configured.

The selected table is saved for notebook 03, keeping geocoding and export
consistent. Each facility is geocoded once. CSV address conflicts raise an
error. Workbook mode uses the configured address worksheet as its canonical
source and checks coverage for every selected facility.


In [ ]:
service_types = None
use_workbook_selection = False

available_service_types = services_clean["service_type"].value_counts()
print("Available service categories:")
print(available_service_types)

selection_input = services_clean
if use_workbook_selection:
    if INPUTS["selection_workbook"] is None:
        raise ValueError("Set inputs.selection_workbook in config.toml first.")
    selection_input = pd.read_excel(
        INPUTS["selection_workbook"],
        sheet_name=INPUTS["service_sheet"],
        dtype={"Equipment ID": "string"},
    )
    selection_input = clean_services(selection_input)

selected_services = select_services(selection_input, service_types)
selected_services.to_csv(INTERIM / "services_selected.csv", index=False)

address_records = selected_services
if use_workbook_selection:
    address_records = pd.read_excel(
        INPUTS["selection_workbook"],
        sheet_name=INPUTS["address_sheet"],
        dtype={"Equipment ID": "string"},
    )
    address_records = standardize_columns(address_records)
    selected_ids = selected_services["equipment_id"]
    address_records = address_records[address_records["equipment_id"].isin(selected_ids)]

addresses = prepare_addresses(address_records)
if len(addresses) != selected_services["equipment_id"].nunique():
    raise ValueError("The address source does not cover every selected facility.")
print("Selected service records:", len(selected_services))
print("Selected service types:", selected_services["service_type"].nunique())
print("Equipment addresses:", len(addresses))
print("Postal-code format issues:", (~addresses["postal_code_valid"]).sum())
addresses.head()


## Coordinate-cache matching

The optional configured coordinate cache provides a reusable input. New
geocoding results are stored separately in interim/geocoding_cache.json.

Reuse requires both equipment ID and address to match after case and whitespace
normalization. Records without precision metadata are marked unknown.
A cache match establishes coordinate availability, not positional accuracy.


In [ ]:
cache_files = [
    INTERIM / "geocoding_cache.json",
]
if INPUTS["coordinate_cache"] is not None:
    cache_files.append(INPUTS["coordinate_cache"])
coordinate_cache = load_coordinate_cache(cache_files)
print("Available cached coordinate pairs:", len(coordinate_cache))


## Geocoding configuration

provider = None keeps execution offline. Select nominatim or google to enable
new requests. Credentials and HTTP identity are read from root .env, loaded
by this notebook before os.getenv is used.

NOMINATIM_URL identifies the instance. NOMINATIM_USERNAME and NOMINATIM_PASSWORD
optionally enable HTTP Basic authentication on a private instance.
GOOGLE_MAPS_API_KEY is required only for new Google requests.

The initial budget is five HTTP calls, including any alternative Nominatim
endpoint request. There is no automatic postal-centroid fallback or switch
to a paid provider. The public Nominatim service has a
[usage policy](https://operations.osmfoundation.org/policies/nominatim/);
use a suitable provider or local instance for a full collection.


In [ ]:
provider = None
google_api_key = os.getenv("GOOGLE_MAPS_API_KEY")

coordinates = geocode_addresses(
    addresses,
    cached=coordinate_cache,
    provider=provider,
    api_key=google_api_key,
    max_requests=5,
    nominatim_url=os.getenv("NOMINATIM_URL") or "http://localhost:8080",
    user_agent=os.getenv("HTTP_USER_AGENT") or "CartaSocialPipeline/1.0",
    nominatim_username=os.getenv("NOMINATIM_USERNAME"),
    nominatim_password=os.getenv("NOMINATIM_PASSWORD"),
)

print(coordinates["status"].value_counts())
coordinates.head()


## Results and unresolved records

All equipment in the selected universe remain in the output, including those
without reusable coordinates. Status `ok` means coordinates are available;
it does not certify positional accuracy. `not_found` indicates an unsuccessful
provider search, whereas `error` indicates a query failure.

The configured cache may cover only part of the collection. Pending
records can reflect an equipment absent from that cache or an address
discrepancy. They are retained for review; expanding the service selection
does not automatically trigger new geocoding requests.

Review unresolved records and their spatial distribution before drawing
territorial conclusions.


In [ ]:
coordinates.to_csv(INTERIM / "equipment_coordinates.csv", index=False)

located_count = (coordinates["status"] == "ok").sum()
unresolved_count = (coordinates["status"] != "ok").sum()

print("Equipment with available coordinates:", located_count)
print("Equipment without a reusable result:", unresolved_count)


## Export handoff

The full cleaned table is stored in `interim/services_clean.csv`.
The configured selection is stored in `interim/services_selected.csv`,
and its equipment coordinates in `interim/equipment_coordinates.csv`.

Continue with [03_geographic_export.ipynb](03_geographic_export.ipynb).
It reads the saved selection rather than imposing its own service categories.
